In [21]:

#Imports
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.features import get_stats, update_team_history, get_h2h_stat, update_h2h
from collections import deque,defaultdict
import joblib

import pandas as pd

sys.path.append(os.path.abspath(".."))



In [22]:
# Load data
df = load_matches() #Load dataframe containing all historical matches
country_elo = load_elo_baseline() #Generate a dictionary with every country having their elo from 12/13/2025

# Prepare and update
recent_matches = prepare_matches(df, start_date="2025-12-13", end_date="2026-06-11") #We need these games since the current elo is only accurate up til 12/13/2025
country_elo = run_elo_updates(recent_matches, country_elo)

#Top 20
top20 = sorted(country_elo.items(), key=lambda x: x[1], reverse=True)[:20]
print("Top 20 teams going into the WC based off ELO")
for rank, (team, rating) in enumerate(top20, 1):
    print(f"{rank}. {team}: {rating:.0f}")

Top 20 teams going into the WC based off ELO
1. Spain: 2163
2. Argentina: 2113
3. France: 2081
4. England: 2020
5. Brazil: 1990
6. Portugal: 1984
7. Colombia: 1977
8. Netherlands: 1944
9. Ecuador: 1937
10. Germany: 1926
11. Norway: 1917
12. Croatia: 1909
13. Japan: 1906
14. Turkey: 1905
15. Switzerland: 1893
16. Uruguay: 1892
17. Belgium: 1888
18. Morocco: 1877
19. Mexico: 1868
20. Italy: 1858


ELO caculations turn out essentially the same to the elo rating found on https://www.eloratings.net/2026_World_Cup, 

In [27]:
current_elos = defaultdict(lambda: 1500)
team_history = defaultdict(lambda: deque(maxlen=10))
h2h_history = defaultdict(lambda: deque(maxlen=5))
training_rows = []


df_sorted = prepare_matches(df, df['date'].min(), pd.Timestamp("2026-06-11")).sort_values('date')
for row in df_sorted.itertuples(index=False):
    
    home_elo = current_elos[row.home_team]
    away_elo = current_elos[row.away_team]
    
    h_form, h_gd = get_stats(row.home_team, team_history)
    a_form, a_gd = get_stats(row.away_team, team_history)
    h2h_val = get_h2h_stat(row.home_team, row.away_team, h2h_history)
        
    training_rows.append({
        'date': row.date,
        'home_team': row.home_team,
        'away_team': row.away_team,
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo-away_elo,
        "home_score": row.home_score,
        "away_score": row.away_score,
        'home_form': h_form,
        'away_form': a_form,
        'h2h': h2h_val,
        'home_gd': h_gd,
        'away_gd': a_gd,
        'neutral': row.neutral,
        'target': row.result
    })
    
    new_away, new_home = update_elo(
        row.result, 
        row.neutral, 
        row.K_factor, 
        row.home_score, 
        row.away_score, 
        current_elos[row.away_team], 
        current_elos[row.home_team]
    )
    
    current_elos[row.home_team] = new_home
    current_elos[row.away_team] = new_away
    
    update_team_history(row, team_history)
    update_h2h(row, h2h_history)
    
# 3. Finalize
train_df = pd.DataFrame(training_rows)
train_df=train_df.round(2)
train_df.to_csv("/Users/armand_k/Projects/WC 2026/data/world_cup_training_data.csv", index=False)



defaultdict(<function load_elo_baseline.<locals>.<lambda> at 0x141cf5e40>, {'Spain': np.float64(2163.4628645456205), 'Argentina': np.float64(2113.4271620560976), 'France': np.float64(2081.118158481849), 'England': np.float64(2019.6773854069866), 'Colombia': np.float64(1976.7364292760565), 'Brazil': np.float64(1989.655734793957), 'Portugal': np.float64(1984.4981931977347), 'Netherlands': np.float64(1944.192642161004), 'Ecuador': np.float64(1936.9654294259358), 'Croatia': np.float64(1909.363843471548), 'Norway': np.float64(1916.8147689670334), 'Germany': np.float64(1925.6371891713034), 'Switzerland': np.float64(1893.4797705606502), 'Uruguay': np.float64(1891.7243225970883), 'Turkey': np.float64(1905.3920054570926), 'Japan': np.float64(1905.9724729356008), 'Denmark': np.float64(1845.8105719807527), 'Italy': np.float64(1858.391607437261), 'Belgium': np.float64(1887.7675481912827), 'Mexico': np.float64(1867.5582430230538), 'Paraguay': np.float64(1834.4370458452204), 'Morocco': np.float64(18

/var/folders/f0/0r4c9w2d0vx1d7zk05qxcr4h0000gn/T/ipykernel_10156/3310334916.py:54: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  train_df=train_df.round(2)


defaultdict(<function load_elo_baseline.<locals>.<lambda> at 0x141cf5e40>, {'Spain': np.float64(2163.4628645456205), 'Argentina': np.float64(2113.4271620560976), 'France': np.float64(2081.118158481849), 'England': np.float64(2019.6773854069866), 'Colombia': np.float64(1976.7364292760565), 'Brazil': np.float64(1989.655734793957), 'Portugal': np.float64(1984.4981931977347), 'Netherlands': np.float64(1944.192642161004), 'Ecuador': np.float64(1936.9654294259358), 'Croatia': np.float64(1909.363843471548), 'Norway': np.float64(1916.8147689670334), 'Germany': np.float64(1925.6371891713034), 'Switzerland': np.float64(1893.4797705606502), 'Uruguay': np.float64(1891.7243225970883), 'Turkey': np.float64(1905.3920054570926), 'Japan': np.float64(1905.9724729356008), 'Denmark': np.float64(1845.8105719807527), 'Italy': np.float64(1858.391607437261), 'Belgium': np.float64(1887.7675481912827), 'Mexico': np.float64(1867.5582430230538), 'Paraguay': np.float64(1834.4370458452204), 'Morocco': np.float64(18

In [28]:
#WC 2026 data
country_elo

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

roster_rows = []

for group_name, teams in wc_groups.items():
    for team in teams:
        elo = country_elo[team]
        form, gd = get_stats(team, team_history)
        
        roster_rows.append({
            "Team": team,
            "Group": group_name.split()[-1],
            "elo": round(elo,2),
            "Avg ppg (L10)": form,
            "Avg gd (L10)": gd,}
        )
roster_df = pd.DataFrame(roster_rows)
roster_df.to_csv("/Users/armand_k/Projects/WC 2026/data/wc_2026_teams.csv", index=False)

defaultdict(<function load_elo_baseline.<locals>.<lambda> at 0x141cf5e40>, {'Spain': np.float64(2163.4628645456205), 'Argentina': np.float64(2113.4271620560976), 'France': np.float64(2081.118158481849), 'England': np.float64(2019.6773854069866), 'Colombia': np.float64(1976.7364292760565), 'Brazil': np.float64(1989.655734793957), 'Portugal': np.float64(1984.4981931977347), 'Netherlands': np.float64(1944.192642161004), 'Ecuador': np.float64(1936.9654294259358), 'Croatia': np.float64(1909.363843471548), 'Norway': np.float64(1916.8147689670334), 'Germany': np.float64(1925.6371891713034), 'Switzerland': np.float64(1893.4797705606502), 'Uruguay': np.float64(1891.7243225970883), 'Turkey': np.float64(1905.3920054570926), 'Japan': np.float64(1905.9724729356008), 'Denmark': np.float64(1845.8105719807527), 'Italy': np.float64(1858.391607437261), 'Belgium': np.float64(1887.7675481912827), 'Mexico': np.float64(1867.5582430230538), 'Paraguay': np.float64(1834.4370458452204), 'Morocco': np.float64(18

In [30]:
print(country_elo)
joblib.dump(dict(country_elo), 'final_elo.joblib')
joblib.dump(dict(team_history), 'final_history.joblib')
joblib.dump(dict(h2h_history), 'final_h2h.joblib')

defaultdict(<function load_elo_baseline.<locals>.<lambda> at 0x141cf5e40>, {'Spain': np.float64(2163.4628645456205), 'Argentina': np.float64(2113.4271620560976), 'France': np.float64(2081.118158481849), 'England': np.float64(2019.6773854069866), 'Colombia': np.float64(1976.7364292760565), 'Brazil': np.float64(1989.655734793957), 'Portugal': np.float64(1984.4981931977347), 'Netherlands': np.float64(1944.192642161004), 'Ecuador': np.float64(1936.9654294259358), 'Croatia': np.float64(1909.363843471548), 'Norway': np.float64(1916.8147689670334), 'Germany': np.float64(1925.6371891713034), 'Switzerland': np.float64(1893.4797705606502), 'Uruguay': np.float64(1891.7243225970883), 'Turkey': np.float64(1905.3920054570926), 'Japan': np.float64(1905.9724729356008), 'Denmark': np.float64(1845.8105719807527), 'Italy': np.float64(1858.391607437261), 'Belgium': np.float64(1887.7675481912827), 'Mexico': np.float64(1867.5582430230538), 'Paraguay': np.float64(1834.4370458452204), 'Morocco': np.float64(18

['final_h2h.joblib']